# RunPod Training

Use this notebook on RunPod, Colab, or any fresh GPU VM. It follows the same Dusty flow as the local training notebook, but keeps the setup self-contained for hosted runtimes.

## Setup

Run this cell in Colab. On RunPod, you can run it too if the workspace is empty. If you already cloned the repo, skip the `git clone` and `%cd` lines and run the install/download commands from the repo root.

In [ ]:
# 1. Clone the repo and enter the directory
!git clone https://github.com/mkhordoo/dusty-llm.git
%cd dusty-llm

# 2. Install uv (takes ~1 second)
!pip install uv

# 3. Install the project using uv into the Colab system environment (takes ~3 seconds)
!uv pip install --system -e .

# 4. Pull the TinyStories and Dusty datasets via your Make command
!make download-datasets

## Check the GPU

Dusty 8M trains on CPU, but a GPU makes the loop faster and is required for larger profiles.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

## Prepare Tokenizer and Training Data

Whenever you change the raw datasets, rerun these preparation cells before training.

In [ ]:
!make dusty-tokenizer
!make dusty-pretrain-data
!make dusty-sft-data

## Smoke-Test Training

`EPOCHS=1` verifies the pipeline. For a useful Dusty checkpoint, train the base model for more tokens, then run SFT for 1-2 epochs.

In [ ]:
!make dusty-pretrain EPOCHS=1 CHECKPOINT_EVERY_STEPS=50
!make dusty-sft-train EPOCHS=1 CHECKPOINT_EVERY_STEPS=50

## Try the SFT Checkpoint

This uses the final SFT checkpoint at `artifacts/checkpoints/dusty8m_sft.pt`.

In [ ]:
!make dusty-generate PROFILE=sft_dusty8m PROMPT="where are you?" TOP_P=0.9 TEMPERATURE=0.8
!make dusty-generate PROFILE=sft_dusty8m PROMPT="what do you dream about?" TOP_P=0.9 TEMPERATURE=0.8

## Keep Training Artifacts

Hosted runtimes can disappear. Download checkpoints or copy them to persistent storage before shutting down the instance.

In [ ]:
from pathlib import Path

for path in sorted(Path("artifacts/checkpoints").glob("*.pt")):
    print(path, f"{path.stat().st_size / 1024 / 1024:.1f} MB")